# Two-period DiD in ModernDiD and R

This notebook is generated from the [Two-period DiD in ModernDiD and R](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_two_period.html) page of the ModernDiD documentation.

This comparison measures how closely [`drdid`](https://moderndid.readthedocs.io/en/stable/api/generated/drdid/moderndid.drdid.html),
[`ipwdid`](https://moderndid.readthedocs.io/en/stable/api/generated/drdid/moderndid.ipwdid.html), and [`ordid`](https://moderndid.readthedocs.io/en/stable/api/generated/drdid/moderndid.ordid.html) reproduce their
counterparts in R's `DRDID` package on the National Supported Work data. Each
pair shows the Python result, the R result, and the numerical difference
between their ATT estimates and standard errors.

The R calls run inside Python through `rpy2` and receive the same observations
and covariates as the Python fits.

> **What these comparisons check**
>
> These are panel specifications with analytical standard errors. The improved
> DR estimator, unnormalized IPW estimator, and outcome regression estimator have
> different assumptions and can return different treatment effects. Here we compare
> each estimator with its own R counterpart.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    # Colab ships rpy2 3.5 as a single package. Uninstalling it first keeps pip from deleting
    # the files it shares with the split rpy2 3.6 packages.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "rpy2"], check=True)
    # rpy2 supports Colab's R 4.6 from release 3.6.7 on.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moderndid[all]==0.4.1", "rpy2>=3.6.7"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-cran-drdid"], check=True)
else:
    print("Outside Colab, set up R as https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running describes.")

## Use the same data in both languages

[`load_nsw`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_nsw.html) provides earnings in 1975 and 1978, a treatment
indicator, and baseline characteristics for 16,417 people observed in both
years. The treatment indicator `experimental` equals one for 425 of them and
zero for the other 15,992. We keep the two-period panel and adjust for age,
education, race, marital status, and whether the participant completed high
school.

The section on [running the comparisons](https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running)
explains how to install the optional R environment and rerun this notebook.

In [ ]:
import rpy2.robjects as ro

import moderndid as did

data = did.load_nsw().select(
    "id", "year", "experimental", "re", "age", "educ", "black", "married", "nodegree", "hisp"
)
formula = "~ age + educ + black + married + nodegree + hisp"

In [ ]:
import numpy as np
import pandas as pd
from rpy2.robjects import pandas2ri

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["nsw"] = ro.conversion.py2rpy(data.to_pandas())

## Improved doubly robust DiD

We set `est_method="imp"` in Python and `estMethod="imp"` in R. Both calls use
a panel, the same covariate formula, and analytical inference. The Python summary
appears immediately below its call.

In [ ]:
python_dr = did.drdid(
    data=data, yname="re", tname="year", idname="id", treatname="experimental",
    xformla=formula, panel=True, est_method="imp", boot=False,
)
print(python_dr)

The R fit uses `dname` for the treatment indicator. Its summary reports the ATT
and standard error from the same observations.

In [ ]:
with ro.default_converter.context():
    r_dr = ro.r("""
    DRDID::drdid(
        data = nsw, yname = "re", tname = "year", idname = "id",
        dname = "experimental",
        xformla = ~ age + educ + black + married + nodegree + hisp,
        panel = TRUE, estMethod = "imp", boot = FALSE
    )
    """)
    _ = ro.r["summary"](r_dr)

Since the summaries round their numbers for display, we extract the underlying
ATT and standard error and subtract them to check agreement beyond that
rounding. The difference columns report Python minus R before any rounding.
The notebook fails if an ATT differs from R by more than
`1e-6 + 1e-6 * abs(R ATT)` or a standard error by more than
`1e-6 + 1e-4 * abs(R SE)`.

In [ ]:
def compare_two_period(python_fit, r_fit, estimator):
    """Set a Python fit beside its R counterpart and check the tolerances."""
    with ro.default_converter.context():
        r_att = r_fit.rx2("ATT")[0]
        r_se = r_fit.rx2("se")[0]
    np.testing.assert_allclose(python_fit.att, r_att, rtol=1e-6, atol=1e-6)
    np.testing.assert_allclose(python_fit.se, r_se, rtol=1e-4, atol=1e-6)
    return pd.DataFrame({
        "estimator": [estimator],
        "Python ATT": [python_fit.att],
        "R ATT": [r_att],
        "ATT difference": [python_fit.att - r_att],
        "Python SE": [python_fit.se],
        "R SE": [r_se],
        "SE difference": [python_fit.se - r_se],
    })


compare_two_period(python_dr, r_dr, "drdid")

## Inverse probability weighting

Unnormalized IPW takes `est_method="ipw"` in Python and `normalized=FALSE` in R.
Since changing either setting changes the estimator, check it when you
translate an existing analysis. Both summaries below confirm the choice by
naming Horvitz-Thompson-type weights.

In [ ]:
python_ipw = did.ipwdid(
    data=data, yname="re", tname="year", idname="id", treatname="experimental",
    xformla=formula, panel=True, est_method="ipw", boot=False,
)
print(python_ipw)

In [ ]:
with ro.default_converter.context():
    r_ipw = ro.r("""
    DRDID::ipwdid(
        data = nsw, yname = "re", tname = "year", idname = "id",
        dname = "experimental",
        xformla = ~ age + educ + black + married + nodegree + hisp,
        panel = TRUE, normalized = FALSE, boot = FALSE
    )
    """)
    _ = ro.r["summary"](r_ipw)

In [ ]:
compare_two_period(python_ipw, r_ipw, "ipwdid")

## Outcome regression

The final pair uses outcome regression on the same sample and covariates. As
above, the estimates shown in the comparison come directly from these fits.

In [ ]:
python_or = did.ordid(
    data=data, yname="re", tname="year", idname="id", treatname="experimental",
    xformla=formula, panel=True, boot=False,
)
print(python_or)

In [ ]:
with ro.default_converter.context():
    r_or = ro.r("""
    DRDID::ordid(
        data = nsw, yname = "re", tname = "year", idname = "id",
        dname = "experimental",
        xformla = ~ age + educ + black + married + nodegree + hisp,
        panel = TRUE, boot = FALSE
    )
    """)
    _ = ro.r["summary"](r_or)

In [ ]:
compare_two_period(python_or, r_or, "ordid")

The paired results show the size of the remaining numerical differences for each
specification. These checks cover the panel estimators shown here and do not
establish agreement for repeated cross-sections or bootstrap inference.

## Repeated cross-sections

With `panel=False` and covariates, the `"trad"` method of [`drdid`](https://moderndid.readthedocs.io/en/stable/api/generated/drdid/moderndid.drdid.html)
and [`drdid_trad_rc`](https://moderndid.readthedocs.io/en/stable/api/generated/drdid/moderndid.drdid_trad_rc.html) give the same estimates as `DRDID` and
differ by design in their standard errors. Since both estimators measure changes
from the pre-treatment period, the pre-treatment averages enter those changes
with a minus sign. The estimation effect of the comparison group's
pre-treatment regression reaches the influence function through those averages
and keeps their sign. `DRDID` 1.3.0 adds that effect instead (`R/drdid_rc.R`,
lines 331 and 332, and `R/drdid_rc1.R`, lines 195 and 220). A finite-difference
derivative of the estimate confirms the minus sign. `R/drdid_rc.R` uses that
sign itself for the matching term of its locally efficient adjustment at line
334. On the National Supported Work data treated as repeated cross-sections,
with age, education, race, and marital status as covariates, the `"trad"`
standard error is 492.32 in ModernDiD and 494.00 in `DRDID`.

## Designs the regressions can't identify

Unlike the National Supported Work data, some designs leave a regression
unidentified because one of its groups has fewer observations than
coefficients or collinear covariates. ModernDiD's two-period estimators then
stop with an error that names the group. The improved estimators of `DRDID` 1.3.0 still return a number by
setting the coefficients they cannot identify to zero.

> **DRDID reports an undefined estimate**
>
> Since the number that `DRDID` returns can change when you reorder or rescale
> the covariate columns, the method defines no such estimate. Rather than report
> it, change the covariates as ModernDiD's error message suggests.

## Versions used

The outputs above were produced with the following versions.

In [ ]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = ro.r("R.version.string")[0]
    drdid_version = ro.r('as.character(packageVersion("DRDID"))')[0]
pd.DataFrame({
    "component": ["Python", "ModernDiD", "rpy2", "R", "DRDID"],
    "version": [python_version(), version("moderndid"), version("rpy2"), r_version, drdid_version],
})